# Notebook 03: Baseline Lasso Model with leave-one-subject-out (LOSO) Cross-Validation

Here we will concatenate the `sub-**_features.csv` items, perform the global `KNNImputer` and `PCA` to finalize `Target B` labels across our entire dataset, and then test the generalizability of a Lasso (L1 Regularized) model to an unseen subject.

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from tqdm import tqdm

base_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
features_dir = os.path.join(base_path, 'data', 'features')

csv_files = glob.glob(os.path.join(features_dir, 'sub-*_features.csv'))
df_list = []
for f in csv_files:
    df_list.append(pd.read_csv(f))

full_df = pd.concat(df_list, ignore_index=True)
print(f"Total trials collected: {len(full_df)}")
full_df.head()

Total trials collected: 990


,track_id,FP1_hjorth_mobility,FP1_hjorth_complexity,FP1_delta_power,FP1_theta_power,FP1_alpha_power,FP1_beta_power,FP1_gamma_power,FP2_hjorth_mobility,FP2_hjorth_complexity,...,rms_energy_mean,valence,energy,tension,anger,fear,happy,sad,tender,TARGET
0,62,0.134352,1.729703,3.407279e-11,1.162846e-11,1.700672e-11,3.895831e-11,2.944737e-11,0.132419,1.754594,...,0.072568,6.83,4.33,2.50,1.00,1.00,2.2,2.8,7.60,TENDER
1,163,0.135765,1.726686,4.352424e-11,1.117196e-11,2.338304e-11,4.184723e-11,3.340625e-11,0.125188,1.862795,...,0.046608,3.60,4.20,5.20,1.00,1.00,2.0,2.5,1.67,SURPRISE
2,160,0.090367,2.529424,9.784519e-11,2.023138e-11,2.085790e-11,3.973028e-11,2.533906e-11,0.067383,3.375198,...,0.058488,5.60,4.60,3.80,1.00,1.00,5.0,1.5,1.83,SURPRISE
3,140,0.084414,2.616659,1.102087e-10,2.152337e-11,2.374664e-11,4.068033e-11,2.173467e-11,0.067030,3.318520,...,0.129327,3.40,5.40,6.20,5.17,4.50,1.0,1.0,1.00,ANGER
4,113,0.074725,3.117129,1.126839e-10,1.598543e-11,2.688764e-11,3.082733e-11,2.540161e-11,0.059408,3.947400,...,0.050776,2.50,5.50,5.83,3.67,4.17,1.0,1.0,1.00,FEAR


### 1. Global Imputation & PCA for Target B

Since we merge 31 subjects here, we run a single master `KNNImputer` across the entire space to reconstruct the continuous subjective emotion values without rotating the PC axes independently per person.

In [2]:
# Subjective Rating Columns
q_cols = [f'Q{q}' for q in range(800, 808)]

# Impute missing Likert scale inputs across all subjects
imputer = KNNImputer(n_neighbors=5, weights='distance')
imputed_ratings = imputer.fit_transform(full_df[q_cols])

# Apply PCA
pca = PCA(n_components=3)
target_b_pca = pca.fit_transform(imputed_ratings)

full_df['PC1_Valence_Subj'] = target_b_pca[:, 0]
full_df['PC2_Energy_Subj'] = target_b_pca[:, 1]
full_df['PC3_Tension_Subj'] = target_b_pca[:, 2]

print(f"Explained Variance by 3 PCs: {sum(pca.explained_variance_ratio_):.2f}")

Explained Variance by 3 PCs: 0.44


/tmp/ipykernel_51885/489580759.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_df['PC1_Valence_Subj'] = target_b_pca[:, 0]
/tmp/ipykernel_51885/489580759.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  full_df['PC2_Energy_Subj'] = target_b_pca[:, 1]
/tmp/ipykernel_51885/489580759.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fr

### 2. Defining the Predictive Features (`X`)
We drop the target mapping components, metadata, and subjective columns to isolate pure EEG + Acoustic features.

In [3]:
drop_cols = ['track_id', 'run', 'subject'] + q_cols + \
            ['valence', 'energy', 'tension', 'anger', 'fear', 'happy', 'sad', 'tender', 'TARGET'] + \
            ['PC1_Valence_Subj', 'PC2_Energy_Subj', 'PC3_Tension_Subj']

X = full_df.drop(columns=drop_cols)
print(f"Feature matrix shape (X): {X.shape}")

Feature matrix shape (X): (990, 199)


### 3. Training: Leave-One-Subject-Out (LOSO) cross-validation
We regress on the Target B subjective PCs to mimic Krish (2021) directly.

In [4]:
subjects = full_df['subject'].unique()
predictions = []
actuals = []

for test_sub in tqdm(subjects, desc="LOSO Fold"):
    # Split
    train_idx = full_df['subject'] != test_sub
    test_idx = full_df['subject'] == test_sub
    
    X_train, y_train = X[train_idx], full_df.loc[train_idx, 'PC1_Valence_Subj']
    X_test, y_test = X[test_idx], full_df.loc[test_idx, 'PC1_Valence_Subj']
    
    # Scale features
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    # Fit Lasso (alpha = 0.1 for high L1 regularization constraint)
    model = Lasso(alpha=0.1, random_state=42)
    model.fit(X_train_s, y_train)
    
    # Predict
    preds = model.predict(X_test_s)
    
    predictions.extend(preds)
    actuals.extend(y_test.values)

rmse = np.sqrt(mean_squared_error(actuals, predictions))
r2 = r2_score(actuals, predictions)
correlation = np.corrcoef(actuals, predictions)[0, 1]

print(f"\n--- LOSO Validation Results (PC1: Valence) ---")
print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")
print(f"Pearson Correlation (r): {correlation:.4f}")

LOSO Fold: 100%|██████████| 31/31 [00:00<00:00, 86.79it/s]


--- LOSO Validation Results (PC1: Valence) ---
RMSE: 2.4108
R² Score: 0.0245
Pearson Correlation (r): 0.1574


### 4. Krish (2021) Replication: K-Fold (Within-Subject) Validation
To prove the structural difference between "zero-calibration" (LOSO) and "within-subject" validation, we replicate Krish's standard 5-Fold Cross Validation mixing subjects. We also benchmark the correlation metrics ($r$) across target components against Combined, EEG-only, and Audio-only feature sets.

In [7]:
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
import warnings
warnings.filterwarnings('ignore') # ignore lasso convergence warnings for this test

aud_keywords = ['mfcc', 'chroma', 'zcr', 'spectral', 'rms', 'bpm', 'tempo']
aud_cols = [c for c in X.columns if any(k in c for k in aud_keywords)]
eeg_cols = [c for c in X.columns if c not in aud_cols]

feature_sets = {
    'Combined': X.columns.tolist(),
    'EEG Only': eeg_cols,
    'Audio Only': aud_cols
}

targets = {
    'PC1_Valence': 'PC1_Valence_Subj',
    'PC2_Energy': 'PC2_Energy_Subj',
    'PC3_Tension': 'PC3_Tension_Subj'
}

print("--- Krish (2021) Replication: TRUE Within-Subject (Intra-subject) Ridge ---")

results = []
subjects = full_df['subject'].unique()
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for target_name, target_col in targets.items():
    for fset_name, cols in feature_sets.items():
        subject_r_scores = []
        
        for sub in subjects:
            idx = full_df['subject'] == sub
            X_sub = full_df.loc[idx, cols]
            y_sub = full_df.loc[idx, target_col]
            
            # Need at least a few samples to do 5-fold sensibly.
            if len(X_sub) < 10:
                continue
                
            preds = np.zeros(len(y_sub))
            
            for train_idx, test_idx in kf.split(X_sub):
                X_train, X_test = X_sub.iloc[train_idx], X_sub.iloc[test_idx]
                y_train, y_test = y_sub.iloc[train_idx], y_sub.iloc[test_idx]
                
                scaler = StandardScaler()
                X_train_s = scaler.fit_transform(X_train)
                X_test_s = scaler.transform(X_test)
                
                # Small penalty to avoid flatlining when N_samples (32) < N_features (200)
                model = Ridge(alpha=100.0, random_state=42)
                model.fit(X_train_s, y_train)
                
                preds[test_idx] = model.predict(X_test_s)
                
            # strict pearson calculation
            if np.std(preds) > 1e-5:
                r = np.corrcoef(y_sub, preds)[0, 1]
                subject_r_scores.append(r)
            else:
                subject_r_scores.append(0.0) # Model predicted a flat line
                
        mean_r = np.nanmean(subject_r_scores)
        results.append({
            'Target': target_name,
            'Features': fset_name,
            'Pearson_r': mean_r
        })

df_res = pd.DataFrame(results)
pivot_res = df_res.pivot(index='Target', columns='Features', values='Pearson_r')
print("\nMean Subject Pearson Correlation (r):")
print(pivot_res[['Combined', 'EEG Only', 'Audio Only']].round(3))

--- Krish (2021) Replication: TRUE Within-Subject (Intra-subject) Ridge ---

Mean Subject Pearson Correlation (r):
Features     Combined  EEG Only  Audio Only
Target                                     
PC1_Valence    -0.051    -0.058      -0.079
PC2_Energy     -0.044    -0.068      -0.095
PC3_Tension    -0.095    -0.118      -0.034


### 5. Random Split Validation (Testing the "Brain Fingerprint" Hypothesis)
By pooling all 990 epochs and shuffling them randomly (80% train, 20% test), the test set will contain samples from subjects the model has already "seen" in the training set. This allows the model to learn the individual physiological baselines (the "brain fingerprint"), drastically inflating performance compared to the strict cross-subject LOSO validation.

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso

print("--- Brain Fingerprint Test: Global Random Train-Test Split (80/20) with Lasso ---")

random_split_results = []

for target_name, target_col in targets.items():
    y = full_df[target_col]
    for fset_name, cols in feature_sets.items():
        X_subset = full_df[cols]
        
        # Simple 80/20 random split across ALL 990 epochs (No subject boundaries)
        X_train, X_test, y_train, y_test = train_test_split(X_subset, y, test_size=0.2, random_state=42)
        
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)
        
        # Using Lasso with a low alpha to allow it to fit the 199 features
        model = Lasso(alpha=0.01, random_state=42)
        model.fit(X_train_s, y_train)
        
        preds = model.predict(X_test_s)
        
        if np.std(preds) > 1e-5:
            r = np.corrcoef(y_test, preds)[0, 1]
        else:
            r = 0.0
            
        random_split_results.append({
            'Target': target_name,
            'Features': fset_name,
            'Pearson_r': r
        })

df_rs = pd.DataFrame(random_split_results)
pivot_rs = df_rs.pivot(index='Target', columns='Features', values='Pearson_r')
print("\nGlobal Random Split Pearson Correlation (r):")
print(pivot_rs[['Combined', 'EEG Only', 'Audio Only']].round(3))

--- Brain Fingerprint Test: Global Random Train-Test Split (80/20) with Lasso ---

Global Random Split Pearson Correlation (r):
Features     Combined  EEG Only  Audio Only
Target                                     
PC1_Valence     0.218    -0.050       0.275
PC2_Energy      0.114    -0.122       0.257
PC3_Tension     0.234    -0.033       0.289
